In [4]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
import jax.scipy.stats as jstats

In [5]:
def make_nn_params_as_list_of_dicts(
        n_dims_input,
        n_dims_output,
        n_dims_per_hidden_list,
        weight_fill_func=None,
        bias_fill_func=None):
    """
    Create an MLP parameter pytree.

    Each layer is a dictionary:
        {
            "w": weight matrix of shape (input_dim, output_dim),
            "b": bias vector of shape (output_dim,)
        }

    The full network is a list of these dictionaries.
    """
    if weight_fill_func is None:
        weight_fill_func = lambda shape: np.zeros(shape, dtype=np.float32)

    if bias_fill_func is None:
        bias_fill_func = lambda shape: np.zeros(shape, dtype=np.float32)

    dims = [n_dims_input] + list(n_dims_per_hidden_list) + [n_dims_output]

    nn_params = []

    for layer_id in range(len(dims) - 1):
        n_in = dims[layer_id]
        n_out = dims[layer_id + 1]

        layer = {
            "w": jnp.asarray(weight_fill_func((n_in, n_out)), dtype=jnp.float32),
            "b": jnp.asarray(bias_fill_func((n_out,)), dtype=jnp.float32),
        }

        nn_params.append(layer)

    return nn_params


def predict_f_given_x(nn_params, x_ND):
    """
    Forward pass through an MLP.

    Uses tanh activation for hidden layers.
    Uses identity activation for output layer.

    Args
    ----
    nn_params : list of dicts
    x_ND : array, shape (N, D)

    Returns
    -------
    f_NK : array, shape (N, K)
    """
    h = x_ND

    for layer_id, layer in enumerate(nn_params):
        w = layer["w"]
        b = layer["b"]

        h = jnp.dot(h, w) + b

        is_hidden_layer = layer_id < len(nn_params) - 1

        if is_hidden_layer:
            h = jnp.tanh(h)

    return h


def pretty_print_nn_param_list(nn_params, prefix=""):
    """
    Small debugging helper.
    """
    for layer_id, layer in enumerate(nn_params):
        print(f"{prefix}layer {layer_id}")
        print("  w shape:", layer["w"].shape)
        print("  b shape:", layer["b"].shape)
        print("  w mean/std:", float(jnp.mean(layer["w"])), float(jnp.std(layer["w"])))
        print("  b mean/std:", float(jnp.mean(layer["b"])), float(jnp.std(layer["b"])))

In [6]:
def softplus_inverse(x):
    """
    Inverse of softplus for positive x.
    """
    x = np.asarray(x)
    return np.log(np.exp(x) - 1.0)


def zeros_like_pytree(pytree):
    return jax.tree.map(lambda x: jnp.zeros_like(x), pytree)


def softplus_of_pytree(pytree):
    return jax.tree.map(lambda x: jax.nn.softplus(x), pytree)